# AI Incidents Pipeline Runner

Loads the configuration from `config/params.yaml` and runs the full pipeline:
ingestion -> preprocessing -> NLP -> sentiment -> analysis -> visualization/report.

In [1]:
from pathlib import Path

import yaml

from src import configure_logging, set_global_seeds
from src.ingestion import load_and_validate
from src.preprocessing import preprocess
from src.nlp import process_text
from src.sentiment import run_sentiment
from src.analysis import run_analysis
from src.visualization import run_visualization

In [2]:
with open("config/params.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

configure_logging()
set_global_seeds(config["random_state"])

In [3]:
df = load_and_validate(config)
df.head()

2026-06-15 00:12:12,124 [INFO] src.ingestion: Loaded 10372 rows from data\raw\oecd_aim.csv (encoding=utf-8)
2026-06-15 00:12:12,133 [INFO] src.ingestion: Column 'text_data' has 0.9% null values
2026-06-15 00:12:12,161 [INFO] src.ingestion: No duplicate rows found


,id,title,text_data,evidences,country,tags_list,industries,harmed,harmlevel,harmtype,threat,event_date,geo_zone
0,4,Clues to Future Snowden Leaks Found In His Past,Only a tiny fraction of Snowden's documents ha...,['A new article by investigative reporter Chri...,USA,"['Respect of human rights', 'Transparency & ex...","['Digital security', 'Government, security and...",['General public'],[],['Psychological'],True,2014-01-02,North America
1,7,"'Talking Angela' programmer talks hoaxes, AI m...","If you're not a parent or teenager, it's possi...","['In fact, Talking Angela is a hugely popular ...",NaN,"['Respect of human rights', 'Robustness & digi...",['Digital security'],['Unknown'],['Non-physical harm'],['Reputational'],False,2014-03-01,Unknown
2,10,Technology: Rise of the replicants,Rapid advances in artificial intelligence now ...,['Rapid advances in artificial intelligence no...,USA,"['Performance', 'Reskill or upskill']","['Financial and insurance services', 'IT infra...",['Workers'],[],['Economic/Property'],True,2014-03-04,North America
3,18,"WeChat, Microsoft trade fire over 'killing' of...",China's popular mobile messaging application W...,"[""WeChat, Microsoft trade fire over 'killing' ...",CHN,"['Robustness & digital security', 'Privacy & d...","['IT infrastructure and hosting', 'Robots, sen...",['Unknown'],['Non-physical harm'],['Reputational'],False,2014-06-02,Asia
4,22,Is the Turing Test milestone all it's cracked ...,Horrific accident kills Utah toddler on visit ...,['A computer program mimicked a conversation a...,USA,"['Transparency & explainability', 'Robustness ...","['IT infrastructure and hosting', 'Robots, sen...",[],[],[],False,2014-06-10,North America


In [4]:
df = preprocess(df, config)
df.head()

2026-06-15 00:12:13,011 [INFO] src.preprocessing: Dropping 3164 rows outside year range [2014, 2023]
2026-06-15 00:12:13,027 [INFO] src.preprocessing: Dropping 2420 rows outside regions ['North America', 'Europe', 'Asia']


,id,title,text_data,evidences,country,tags_list,industries,harmed,harmlevel,harmtype,...,mlb_harmlevel__Hazard,mlb_harmlevel__Injury,mlb_harmlevel__Non-physical harm,mlb_harmtype__Economic/Property,mlb_harmtype__Human rights,mlb_harmtype__Physical,mlb_harmtype__Psychological,mlb_harmtype__Public interest,mlb_harmtype__Reputational,mlb_harmtype__Unknown
0,4,Clues to Future Snowden Leaks Found In His Past,Only a tiny fraction of Snowden's documents ha...,['A new article by investigative reporter Chri...,USA,"[Respect of human rights, Transparency & expla...","[Digital security, Government, security and de...",[General public],[],[Psychological],...,0,0,0,0,0,0,1,0,0,0
2,10,Technology: Rise of the replicants,Rapid advances in artificial intelligence now ...,['Rapid advances in artificial intelligence no...,USA,"[Performance, Reskill or upskill]","[Financial and insurance services, IT infrastr...",[Workers],[],[Economic/Property],...,0,0,0,1,0,0,0,0,0,0
3,18,"WeChat, Microsoft trade fire over 'killing' of...",China's popular mobile messaging application W...,"[""WeChat, Microsoft trade fire over 'killing' ...",CHN,"[Robustness & digital security, Privacy & data...","[IT infrastructure and hosting, Robots, sensor...",[Unknown],[Non-physical harm],[Reputational],...,0,0,1,0,0,0,0,0,1,0
4,22,Is the Turing Test milestone all it's cracked ...,Horrific accident kills Utah toddler on visit ...,['A computer program mimicked a conversation a...,USA,"[Transparency & explainability, Robustness & d...","[IT infrastructure and hosting, Robots, sensor...",[],[],[],...,0,0,0,0,0,0,0,0,0,0
6,34,More Privacy Woes For Google: This Time It's A...,Another feature of Google Search is under chal...,['A Hong Kong court today ruled that local bus...,HKG,"[Fairness, Accountability, Transparency & expl...",[Digital security],[Business],[Non-physical harm],"[Economic/Property, Human rights, Public inter...",...,0,0,1,1,1,0,0,1,1,0


In [5]:
df = process_text(df, config)
df[["tokens", "mental_health_flag"]].head()

2026-06-15 00:12:13,138 [INFO] src.nlp: Downloading NLTK resource 'wordnet'
2026-06-15 00:12:13,379 [INFO] src.nlp: Downloading NLTK resource 'omw-1.4'


,tokens,mental_health_flag
0,"[tiny, fraction, snowdens, document, published...",0
2,"[rapid, advance, artificial, intelligence, thr...",0
3,"[china, popular, mobile, messaging, applicatio...",0
4,"[horrific, accident, kill, utah, toddler, visi...",0
6,"[another, feature, google, search, challenge, ...",0


In [6]:
df = run_sentiment(df, config)
df[["sentiment_score", "sentiment_label"]].head()

c:\Users\USER\Documents\ucom\tesis\ai-incidents-pipeline\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\USER\Documents\ucom\tesis\ai-incidents-pipeline\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--cardiffnlp--twitter-roberta-base-sentiment-latest. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activat

,sentiment_score,sentiment_label
0,0.075981,Baja
2,0.770157,Alta
3,0.052453,Baja
4,0.125906,Baja
6,0.449286,Media


In [7]:
processed_path = Path(config["data"]["processed_path"])
processed_path.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(processed_path)

In [8]:
metrics = run_analysis(df, config)

2026-06-15 00:32:37,312 [INFO] src.analysis: Metrics written to outputs\reports\metrics.json


In [9]:
report_path = run_visualization(df, metrics, config)
report_path

2026-06-15 00:32:38,160 [INFO] src.visualization: Saved figure 'regional_evolution' to outputs\figures\regional_evolution.png
2026-06-15 00:32:38,601 [INFO] src.visualization: Saved figure 'cumulative_concentration' to outputs\figures\cumulative_concentration.png
2026-06-15 00:32:39,122 [INFO] src.visualization: Saved figure 'principles_distribution' to outputs\figures\principles_distribution.png
2026-06-15 00:32:39,631 [INFO] src.visualization: Saved figure 'harm_types_distribution' to outputs\figures\harm_types_distribution.png
2026-06-15 00:32:40,290 [INFO] src.visualization: Saved figure 'stakeholders_distribution' to outputs\figures\stakeholders_distribution.png
2026-06-15 00:32:40,582 [INFO] src.visualization: Saved figure 'vulnerable_groups' to outputs\figures\vulnerable_groups.png
2026-06-15 00:32:41,136 [INFO] src.visualization: Saved figure 'harm_chronology' to outputs\figures\harm_chronology.png
2026-06-15 00:32:44,125 [INFO] src.visualization: Saved figure 'negativity_by_pr

WindowsPath('outputs/reports/executive_report.pdf')

In [10]:
print("Pipeline completado.")

Pipeline completado.
